# Autoresearch Experiment Analysis

Analysis of autonomous PPO hyperparameter tuning results on CarRacing-v3 from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, avg_return, memory_gb, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["avg_return"] = pd.to_numeric(df["avg_return"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    ret = row["avg_return"]
    desc = row["description"]
    print(f"  #{i:3d}  avg_return={ret:.2f}  mem={row['memory_gb']:.1f}GB  {desc}")

## Avg Return Over Time

Track how the best (kept) avg_return evolves as experiments progress. The running maximum shows the "frontier" -- the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_ret = valid.loc[0, "avg_return"]

# Plot discarded as faint background dots
disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["avg_return"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["avg_return"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running maximum step line
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_ret = valid.loc[kept_mask, "avg_return"]
running_max = kept_ret.cummax()
best = running_max.iloc[-1]
ax.step(kept_idx, running_max, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment with its description
for idx, ret in zip(kept_idx, kept_ret):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."

    ax.annotate(desc, (idx, ret),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Avg Return (higher is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

# Y-axis: from just below baseline to just above best
margin = (best - baseline_ret) * 0.15
ax.set_ylim(baseline_ret - margin, best + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_ret = df.iloc[0]["avg_return"]
best_ret = kept["avg_return"].max()
best_row = kept.loc[kept["avg_return"].idxmax()]

print(f"Baseline avg_return:  {baseline_ret:.2f}")
print(f"Best avg_return:      {best_ret:.2f}")
print(f"Total improvement:    {best_ret - baseline_ret:.2f}")
print(f"Best experiment:      {best_row['description']}")
print()

# How many experiments to find each improvement
print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: avg_return={row['avg_return']:.2f}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment's avg_return
# (since experiments are cumulative -- each one builds on the last kept state)
kept = df[df["status"] == "KEEP"].copy()
kept["prev_ret"] = kept["avg_return"].shift(1)
kept["delta"] = kept["avg_return"] - kept["prev_ret"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>9}  {'Return':>10}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+9.2f}  {row['avg_return']:10.2f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+9.2f}  {'':>10}  TOTAL improvement over baseline")

## Export GIF of Best Policy

Loads the current trained model (from the latest `train.py`) and records one episode as a GIF.

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np
from PIL import Image

# Import the model class and preprocessing from train.py
from train import ActorCritic, postprocess_action

# --- Load the best model by re-running train.py's model definition ---
# We reconstruct the model architecture and load the EMA weights from the
# last training run. Since train.py executes on import, we use the already-
# loaded ema_model from the module.
from train import ema_model, device

ema_model.eval()

# --- Record one episode ---
env = gym.make("CarRacing-v3", render_mode="rgb_array")
obs, _ = env.reset(seed=2024)
frames = []
episode_return = 0.0
done = False

while not done:
    frame = env.render()
    frames.append(frame)

    # Preprocess single obs: (96,96,3) -> (1,3,96,96)
    obs_p = obs.astype(np.float32) / 255.0
    obs_p = np.transpose(obs_p, (2, 0, 1))
    obs_t = torch.as_tensor(obs_p, dtype=torch.float32, device=device).unsqueeze(0)

    with torch.no_grad():
        action = ema_model.get_deterministic_action(obs_t)
    action_np = postprocess_action(action.squeeze(0).cpu().numpy())

    obs, reward, terminated, truncated, _ = env.step(action_np)
    episode_return += reward
    done = terminated or truncated

env.close()
print(f"Episode return: {episode_return:.1f}")
print(f"Frames captured: {len(frames)}")

In [ ]:
# Save as GIF
pil_frames = [Image.fromarray(f) for f in frames]
gif_path = "carracing_policy.gif"
pil_frames[0].save(
    gif_path,
    save_all=True,
    append_images=pil_frames[1:],
    duration=1000 // 50,  # 50 FPS playback (CarRacing runs at 50 FPS)
    loop=0,
)
print(f"Saved {gif_path} ({len(pil_frames)} frames, {len(pil_frames) / 50:.1f}s)")

# Display in notebook
from IPython.display import Image as IPImage, display
display(IPImage(filename=gif_path))